# Punto 2 — LGD: referencia, red neuronal y pérdida esperada

Este notebook usa **la red de Manuel** (`src/red_neuronal.py`). Incluye los cinco apartados del punto 2 y la regla de comparación justa. Ejecute las celdas en orden con Python 3.12 y el entorno `.venv`. Las tablas y figuras se guardan para regenerarse sin reentrenar. No hay resultados crediticios precalculados en este notebook.

Seleccione en VS Code el kernel de `.venv`. Si no aparece, instale en la terminal: `.\\.venv\\Scripts\\python.exe -m pip install ipykernel`.

El modo completo usa 30 semillas; la demostración usa 3 y **no cumple por sí sola** ese requisito. Para un experimento desde terminal: `python -m src.experimentos_lgd --epocas 200`; añada `--demo` para una corrida corta. Mida su duración en su PC.


In [ ]:
from pathlib import Path
import sys
import json

# Permite abrir el notebook desde la raíz o desde Punto_2/Modelos.
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/red_neuronal.py').exists()), None)
if RAIZ is None:
    raise FileNotFoundError('Abra el notebook dentro del repositorio')
sys.path.insert(0, str(RAIZ))
import os
os.chdir(RAIZ)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Configuración editable, sin cambiar la implementación de los modelos.
DEMO = True  # Cambiar a False para el experimento completo de 30 semillas.
EPOCAS = 60 if DEMO else 200
SEMILLAS = list(range(3 if DEMO else 30))
CSV = RAIZ / 'Punto_2/loan.csv'
DATOS = RAIZ / 'data/processed/lgd_limpio.parquet'
SALIDA = RAIZ / ('reports/punto2_demo' if DEMO else 'reports/punto2_completo')
CANDIDATOS = [
    dict(capas_ocultas=(16,), activacion='tanh', l2=0.0001, dropout=0., tasa=.001),
    dict(capas_ocultas=(32,16), activacion='relu', l2=0.0001, dropout=.1, tasa=.001),
    dict(capas_ocultas=(32,16), activacion='tanh', l2=.001, dropout=.1, tasa=.003),
]
print('DEMOSTRACIÓN, no resultados finales' if DEMO else 'EXPERIMENTO COMPLETO')


## 1. Variable objetivo: fórmula, campos y limitaciones

$$LGD_{proxy,i}=1-\frac{total\_rec\_prncp_i}{funded\_amnt_i}.$$

`funded_amnt` es el monto original financiado y `total_rec_prncp` es el capital recibido a la fecha. El diccionario `LCDataDictionary.xlsx` describe este último como *Principal received to date*. La proxy mide capital original no cubierto por ese campo a la fecha de observación. No incorpora explícitamente `recoveries − collection_recovery_fee`, costos integrales ni flujos descontados. No es LGD regulatoria exacta; el enunciado permite construir una medida, pero exige justificarla.

Se conserva la muestra `Charged Off` y `Default` de los experimentos previos. **No equivale a toda la clase 1 de PD del enunciado**, que también incluye `Late (31-120 days)` y `Does not meet the credit policy. Status:Charged Off`. Para representar a todos los incumplidos hay que justificar la restricción o ampliar y volver a preparar/entrenar. Las recuperaciones pueden estar censuradas, especialmente en `Default`.

El script comprueba monto positivo, capital recibido finito/no negativo y LGD en [0,1]. No recorta registros inválidos silenciosamente. Los componentes ex post del objetivo no entran como predictores. Verifique que el CSV corresponde al dataset del enunciado y documente fecha de descarga y periodo; no se infieren del nombre del archivo.


In [ ]:
from preparar_lgd import COLUMNAS, preparar

if CSV.exists():
    bruto = pd.read_csv(CSV, usecols=COLUMNAS, low_memory=False)
    print('Filas del CSV:', len(bruto))
    display(bruto['loan_status'].value_counts().rename('n').to_frame())
    preparado = preparar(bruto, semilla=42)
    DATOS.parent.mkdir(parents=True, exist_ok=True)
    preparado.to_parquet(DATOS, index=False)
    procedencia = dict(formula='1 - total_rec_prncp / funded_amnt',
                      poblacion=['Charged Off','Default'], origen=str(CSV),
                      semilla_particion=42, particion='aleatoria 60/20/20')
    DATOS.with_suffix('.metadata.json').write_text(json.dumps(procedencia, indent=2), encoding='utf-8')
elif DATOS.exists() and DATOS.with_suffix('.metadata.json').exists():
    preparado = pd.read_parquet(DATOS)
    procedencia = json.loads(DATOS.with_suffix('.metadata.json').read_text(encoding='utf-8'))
    if procedencia.get('formula') != '1 - total_rec_prncp / funded_amnt':
        raise ValueError('El objetivo del parquet no coincide con la proxy acordada')
    print('Se reutilizan datos procesados; consulte sus metadatos de procedencia.')
else:
    raise FileNotFoundError('Falta Punto_2/loan.csv o el parquet con sus metadatos')
print(procedencia)
display(preparado.groupby('particion')['lgd'].agg(['count','mean','std','min','max']))


## 2. Referencia y comparación justa

La referencia es una **regresión lineal OLS**, ajustada con `LinearRegression`. Se evalúan sus predicciones sin recorte; así queda visible si salen de [0,1]. El número nominal de parámetros es p+1 (p columnas transformadas más intercepto); con one-hot completo puede haber dependencia lineal y ese número no representa rango efectivo.

OLS y todas las redes comparten filas, objetivo, particiones y **el mismo preprocesador** ajustado solo en entrenamiento: imputación, log1p de ingresos/saldo, estandarización y one-hot. Las particiones son aleatorias 60/20/20, fijas entre semillas; no se presentan como evaluación temporal. No se reutilizan como comparación directa los resultados antiguos del notebook lineal con otra partición.

Presupuesto explícito: 1 configuración OLS; 3 configuraciones neuronales (editables arriba), cada una con semilla 42 para búsqueda. Se elige por menor MAE en validación. Luego se repite esa configuración con 30 semillas distintas en modo completo. OLS no requiere repeticiones por inicialización porque es determinista en este flujo. Se informan por separado tiempos de referencia, búsqueda y réplicas; los presupuestos de las dos familias son distintos.

## 3. Red neuronal y elección de salida/pérdida

La red de Manuel usa capas densas, ReLU o tanh, Adam, L2, dropout y parada temprana con restauración de la mejor época de validación. La salida es sigmoide, $\sigma(z)=1/(1+e^{-z})$, porque la proxy está acotada. Se minimiza MSE, $(1/n)\sum_i(y_i-\hat y_i)^2$, para estimar una media condicional; no se trata LGD como etiqueta binaria. La sigmoide puede aproximar, pero no alcanzar en aritmética exacta, los extremos 0 y 1. MSE coincide, salvo escala/constantes, con una pérdida gaussiana de varianza fija; aquí se justifica como error de predicción, sin afirmar soporte gaussiano de LGD.

La búsqueda compara arquitecturas, activación, L2, dropout y tasa bajo un presupuesto pequeño y reproducible. Adam, batch size y paciencia quedan fijos: no se afirma haber buscado todos los hiperparámetros posibles. No se cambia ninguna decisión a partir de prueba.


In [ ]:
from src.experimentos_lgd import ejecutar

resultado = ejecutar(ruta=DATOS, salida=SALIDA, demo=DEMO, epocas=EPOCAS,
                     semillas=SEMILLAS, candidatos=CANDIDATOS)
display(resultado['busqueda'].rename_axis('Tabla 1: búsqueda por validación'))
display(resultado['referencia'].rename_axis('Tabla 2: referencia OLS en prueba'))
display(resultado['corridas'].rename_axis('Tabla 3: red por semilla en prueba'))
display(resultado['resumen'].rename_axis('Tabla 4: resumen de semillas; menor es mejor'))
print(json.dumps(resultado['manifiesto'], indent=2))


## 4. Evaluación: MAE, RMSE y distribución

$$MAE=\frac1n\sum_i|y_i-\hat y_i|,\qquad RMSE=\sqrt{\frac1n\sum_i(y_i-\hat y_i)^2}.$$

Multiplicar por 100 convierte a puntos porcentuales de LGD. RMSE penaliza más los errores grandes. Las Tablas 2–4 proceden de las mismas filas de prueba. La desviación entre semillas usa la desviación muestral (ddof=1). Se reportan media, desviación, mejor (mínimo) y peor (máximo) de MAE, RMSE y distancia descriptiva KS. La prueba se usa para evaluar, no para seleccionar la configuración ni la semilla a desplegar.

Las siguientes figuras comparan OLS y la **primera semilla predeclarada** de la red. Una menor dispersión predicha puede ser coherente con estimar E[LGD|X]; no basta que las medias se parezcan para afirmar que las distribuciones coinciden. Fuente de todas las tablas y figuras: elaboración propia desde los archivos del experimento.


In [ ]:
from src.metricas import comparar_distribuciones

with np.load(SALIDA / 'predicciones_comparadas.npz') as archivo:
    y = archivo['y_test'].copy()
    predicciones = {'OLS': archivo['pred_lineal'].copy(),
                   'Red': archivo[f'pred_semilla_{SEMILLAS[0]}'].copy()}
comparacion = pd.concat({nombre: comparar_distribuciones(y,pred)
                        for nombre,pred in predicciones.items()},axis=1)
comparacion.to_csv(SALIDA / 'tabla_5_distribuciones.csv')
display(comparacion.rename_axis('Tabla 5: distribución observada y predicha'))
print('Predicciones OLS fuera de [0,1]:', int(((predicciones['OLS']<0)|(predicciones['OLS']>1)).sum()))
fig, axes = plt.subplots(1,2,figsize=(12,4))
series = {'Observada': y, **predicciones}
valores = np.concatenate(list(series.values()))
bins = np.linspace(min(0,valores.min()),max(1,valores.max()),31)
for nombre, valores in series.items():
    axes[0].hist(valores,bins=bins,density=True,alpha=.4,label=nombre)
    axes[1].step(np.sort(valores),np.arange(1,len(valores)+1)/len(valores),where='post',label=nombre)
axes[0].set(title='Figura 1a. Distribuciones de LGD',xlabel='LGD',ylabel='Densidad')
axes[1].set(title='Figura 1b. Distribuciones acumuladas',xlabel='LGD',ylabel='Proporción acumulada')
for ax in axes: ax.legend()
fig.tight_layout()
fig.savefig(SALIDA/'figura_1_distribuciones.png',dpi=180)
plt.show()


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(11,4))
with np.load(SALIDA/'historia_red.npz') as historia:
    for clave in historia.files:
        axes[0].plot(np.arange(1,len(historia[clave])+1),historia[clave],label=clave)
axes[0].set(title='Figura 2a. Aprendizaje de la red',xlabel='Época',ylabel='MSE')
axes[0].legend()
for nombre,pred in predicciones.items():
    axes[1].scatter(pred,y-pred,s=5,alpha=.2,label=nombre)
axes[1].axhline(0,color='black')
axes[1].set(title='Figura 2b. Residuos en prueba',xlabel='LGD predicha',ylabel='Observada − predicha')
axes[1].legend()
fig.tight_layout()
fig.savefig(SALIDA/'figura_2_aprendizaje_residuos.png',dpi=180)
plt.show()


## 5. Pérdida esperada: ambos modelos y EAD

$$\widehat{E[L]}_{proxy,i}=\widehat{PD}_i\,\widehat{LGD}_{proxy,i}\,EAD_{proxy,i},\qquad EAD_{proxy,i}=funded\_amnt_i.$$

Se usa capital original para mantener el mismo denominador de la proxy: no se multiplica una severidad normalizada por monto original por un saldo distinto sin redefinirla. `funded_amnt` es conocido una vez financiado; para una decisión anterior debe definirse un monto propuesto como escenario y validar ese uso. No es EAD histórica exacta. `out_prncp` observado puede quedar en cero tras el castigo y no reconstruye esa EAD histórica.

La justificación probabilística es $E[L|X,M]=P(D=1|X,M)E[LGD|D=1,X,M]M$, con M fijo. No exige independencia PD/LGD. Si EAD fuera aleatoria se necesita $PD\,E[LGD\,EAD|D=1,X]$, no separar esperanzas sin justificación.

**Condiciones:** PD debe estimar ser malo (clase 1), la fila debe corresponder al mismo crédito en ambos modelos y los horizontes deben ser coherentes. La LGD actual se limita a Charged Off/Default: extrapolar a toda la clase positiva del enunciado no está validado. Justifique esta restricción o amplíe LGD antes de presentar una pérdida final de toda la población. El resultado no incorpora descuento temporal ni recuperaciones/costos explícitos.

La celda siguiente carga modelos ya entrenados, sin reentrenar PD y sin sustituirla por probabilidades inventadas. Use un CSV/parquet de créditos con `id` único, `funded_amnt` y las entradas tipadas para ambos preprocesadores. No use directamente `loan.csv` crudo ni el parquet LGD que omite id y exposición. Si no tiene el archivo o modelo PD, la celda informa el pendiente y el punto de pérdida esperada **no queda ejecutado**. Para validación fuera de muestra se requiere una cohorte que no se usara para ajustar/seleccionar ninguno de los dos modelos.


In [ ]:
from estimar_perdida import estimar

MODELO_PD = RAIZ / 'models/pd'
MODELO_LGD = SALIDA / 'modelo_red'
CREDITOS = RAIZ / 'datos/creditos_para_predecir.parquet'  # Ajustar al archivo real.
requeridos = [CREDITOS, MODELO_PD/'red.npz', MODELO_PD/'preprocesador.joblib',
              MODELO_LGD/'red.npz', MODELO_LGD/'preprocesador.joblib']
faltantes = [str(p) for p in requeridos if not p.exists()]
if faltantes:
    print('PENDIENTE: no se calculó la pérdida esperada real. Faltan:')
    print('\n'.join(faltantes))
else:
    clientes = (pd.read_parquet(CREDITOS) if CREDITOS.suffix == '.parquet'
                else pd.read_csv(CREDITOS))
    perdidas = estimar(clientes, MODELO_PD, MODELO_LGD)
    perdidas.to_csv(SALIDA/'perdida_esperada_por_credito.csv',index=False)
    display(perdidas.head().rename_axis('Tabla 6: pérdida proxy por crédito'))
    print('Pérdida esperada proxy total (USD):', perdidas.perdida_esperada_proxy.sum())


## 6. Discusión, reproducibilidad y lista de cierre

Complete la discusión con las tablas producidas, sin copiar resultados de otra corrida:

- ¿La red mejora MAE y RMSE respecto a OLS? ¿Esa mejora es consistente entre semillas?
- ¿Cómo se comparan colas, dispersión y sesgo de predicciones en la Figura 1 y Tabla 5? ¿Qué indican los residuos de la Figura 2?
- ¿Qué costo adicional de búsqueda/entrenamiento y complejidad tiene la red? El archivo `experimento.json` registra tiempos, configuraciones, parámetros y filas.
- ¿Qué limitaciones introducen la muestra de incumplimiento, censura de cobros, partición aleatoria y definición proxy? ¿Para qué población y momento puede usarse?

**Artefactos:** `referencia.csv`, `busqueda_validacion.csv`, `corridas.csv`, `resumen_semillas.csv`, tablas/figuras, predicciones, OLS/preprocesador joblib y `modelo_red/red.npz` con preprocesador. El modelo exportado es el de la primera semilla predeclarada, no el mejor en prueba. Los archivos NPZ y figuras permiten volver a evaluar sin entrenar. Cargue solo archivos joblib de procedencia conocida.

Para entregar: ejecutar modo completo con >=30 semillas, registrar fecha/procedencia de datos, guardar datos procesados permitidos y artefactos finales, completar discusión y ejecutar la sección de pérdida esperada real con PD y cohorte compatibles. El notebook no inventa esas evidencias. La comparación temporal y las evaluaciones específicas de PD corresponden al punto 1.

Referencia de interpretación de MSE como estimación de media condicional: Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The elements of statistical learning* (2nd ed.). Springer. https://doi.org/10.1007/978-0-387-84858-7. Fuente de definiciones de campos: `LCDataDictionary.xlsx` incluido; documentar también la procedencia concreta del dataset del equipo.
